# HMCN-F Hyperparameter Sweep — Dataset A

Full grid search over `lambda_viol`, `lr`, `weight_decay`, `dropout`.
`global_dim`, `local_dim`, `beta` fixed at previously tuned values.

## Grid
| Parameter | Values |
|---|---|
| `lambda_viol` | 0, 0.01, 0.05, 0.1, 0.3 |
| `lr` | 1e-3, 5e-4, 1e-4 |
| `weight_decay` | 1e-4, 1e-3 |
| `dropout` | 0.3, 0.47 |

**Total: 5 × 3 × 2 × 2 = 60 runs**

## Crash safety
Each run saves its result to `hmcn_ablation_results.csv` immediately after
evaluation. On restart, already-completed configs are detected and skipped.


## 1. Install dependencies

In [ ]:
!pip install iterative-stratification -q

## 2. Imports and device

In [ ]:
import os
import gc
import json
import itertools
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
import warnings
warnings.filterwarnings('ignore')

# hmcn_eval.py must be uploaded to Colab alongside this notebook
from hmcn_eval import (
    find_optimal_thresholds,
    compute_test_loss,
    compute_all_metrics,
    save_experiment,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## 3. Fixed hyperparameters and grid definition

In [ ]:
# ── Fixed (validated by prior tuning) ────────────────────────────────────────
GLOBAL_DIM  = 128
LOCAL_DIM   = 64
BETA        = 0.5
BATCH_SIZE  = 32
EPOCHS      = 300
PATIENCE    = 40
T_0_EPOCHS  = 50
T_MULT      = 2
ETA_MIN     = 1e-5
SEED        = 42

# ── Sweep grid ────────────────────────────────────────────────────────────────
LAMBDA_VIOL_VALUES  = [0, 0.01, 0.05, 0.1, 0.3]
LR_VALUES           = [1e-3, 5e-4, 1e-4]
WEIGHT_DECAY_VALUES = [1e-4, 1e-3]
DROPOUT_VALUES      = [0.3, 0.47]

# Build full grid
all_configs = [
    {
        'lambda_viol'  : lv,
        'lr'           : lr,
        'weight_decay' : wd,
        'dropout'      : do,
    }
    for lv, lr, wd, do in itertools.product(
        LAMBDA_VIOL_VALUES,
        LR_VALUES,
        WEIGHT_DECAY_VALUES,
        DROPOUT_VALUES,
    )
]

print(f'Total configs : {len(all_configs)}')
print(f'Fixed params  : global_dim={GLOBAL_DIM}, local_dim={LOCAL_DIM}, beta={BETA}')
print(f'Seed          : {SEED}')


## 4. Crash recovery — skip already completed configs

Reads `hmcn_ablation_results.csv` if it exists and identifies which
configs have already been run. Matching is done on the 4 sweep parameters.


In [ ]:
CSV_PATH = 'hmcn_ablation_results.csv'

def get_completed_configs(csv_path):
    """
    Returns a set of (lambda_viol, lr, weight_decay, dropout) tuples
    already present in the results CSV.
    """
    if not os.path.exists(csv_path):
        return set()
    df = pd.read_csv(csv_path)
    # Only consider rows from this sweep experiment
    df = df[df['experiment'] == 'datasetA_sweep']
    completed = set(
        zip(
            df['lambda_viol'].round(4),
            df['lr'].round(6),
            df['weight_decay'].round(6),
            df['dropout'].round(3),
        )
    )
    return completed


def config_key(cfg):
    return (
        round(cfg['lambda_viol'],  4),
        round(cfg['lr'],           6),
        round(cfg['weight_decay'], 6),
        round(cfg['dropout'],      3),
    )


completed = get_completed_configs(CSV_PATH)
remaining  = [c for c in all_configs if config_key(c) not in completed]

print(f'Already completed : {len(completed)}')
print(f'Remaining         : {len(remaining)}')


## 5. Hierarchy definition — Dataset A

In [ ]:
META_CATEGORIES = {
    'floral':        ['rose','jasmin','lily','muguet','violet','hyacinth',
                      'geranium','lavender','orangeflower','chamomile','hawthorn'],
    'fruity':        ['apple','apricot','banana','berry','cherry','grape',
                      'grapefruit','lemon','melon','orange','peach','pear','pineapple',
                      'plum','raspberry','strawberry','tropical','black currant','fruit skin'],
    'sweet':         ['vanilla','caramellic','honey','chocolate','cocoa',
                      'coconut','creamy','buttery','milky','dairy'],
    'woody':         ['cedar','sandalwood','pine','vetiver','terpenic',
                      'balsamic','cortex'],
    'green':         ['grassy','herbal','leafy','hay','tea','fresh',
                      'cucumber','vegetable','weedy'],
    'spicy':         ['cinnamon','clove','warm','pungent','sharp',
                      'cooling','mint','camphoreous'],
    'animal_musk':   ['animal','musk','leathery','fishy','sweaty','meaty',
                      'beefy','musty'],
    'earthy':        ['mushroom','nutty','hazelnut','roasted','coffee',
                      'tobacco','smoky','popcorn'],
    'citrus':        ['bergamot','ozone','clean','soapy'],
    'chemical':      ['solvent','ethereal','metallic','medicinal','phenolic',
                      'sulfurous','gassy','burnt','oily'],
    'gourmand':      ['almond','malty','rummy','brandy','cognac','winey','cooked',
                      'potato','savory','celery','tomato','radish','onion','garlic',
                      'cabbage','cheesy'],
    'powdery_amber': ['amber','powdery','anisic','coumarinic','orris','waxy',
                      'aldehydic','ketonic','lactonic'],
}


## 6. Data loading — run once, reused across all configs

In [ ]:
def load_data(csv_path='hmcn_dataset_A_drop.csv'):
    df = pd.read_csv(csv_path, sep=';')

    fine_cols = [c for c in df.columns if c.startswith('fine_')]
    meta_cols = [c for c in df.columns if c.startswith('meta_')]
    feat_cols = [c for c in df.columns
                 if c not in fine_cols + meta_cols + ['SMILES']]

    stds = df[feat_cols].std()
    feat_cols = stds[stds > 0].index.tolist()

    X          = df[feat_cols].values.astype(np.float32)
    Y1         = df[fine_cols].values.astype(np.float32)
    Y2         = df[meta_cols].values.astype(np.float32)
    fine_names = [c.replace('fine_', '') for c in fine_cols]
    meta_names = [c.replace('meta_', '') for c in meta_cols]

    return X, Y1, Y2, fine_names, meta_names


def split_and_scale(X, Y1, Y2, seed=42):
    msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    trainval_idx, test_idx = next(msss.split(X, Y1))

    msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.1/0.8, random_state=seed)
    train_idx, val_idx = next(msss2.split(X[trainval_idx], Y1[trainval_idx]))

    X_train  = X[trainval_idx][train_idx]
    X_val    = X[trainval_idx][val_idx]
    X_test   = X[test_idx]

    Y1_train = Y1[trainval_idx][train_idx]
    Y1_val   = Y1[trainval_idx][val_idx]
    Y1_test  = Y1[test_idx]

    Y2_train = Y2[trainval_idx][train_idx]
    Y2_val   = Y2[trainval_idx][val_idx]
    Y2_test  = Y2[test_idx]

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)

    return (X_train, Y1_train, Y2_train,
            X_val,   Y1_val,   Y2_val,
            X_test,  Y1_test,  Y2_test)


def build_violation_pairs(fine_names, meta_names):
    fine_idx = {name: i for i, name in enumerate(fine_names)}
    meta_idx = {name: i for i, name in enumerate(meta_names)}
    pairs = []
    for meta, members in META_CATEGORIES.items():
        for member in members:
            if member in fine_idx and meta in meta_idx:
                pairs.append((fine_idx[member], meta_idx[meta]))
    return pairs


def make_loader(X, Y1, Y2, batch_size, shuffle):
    dataset = TensorDataset(
        torch.tensor(X,  dtype=torch.float32),
        torch.tensor(Y1, dtype=torch.float32),
        torch.tensor(Y2, dtype=torch.float32)
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


# ── Load once ─────────────────────────────────────────────────────────────────
X, Y1, Y2, fine_names, meta_names = load_data('hmcn_dataset_A_drop.csv')
violation_pairs = build_violation_pairs(fine_names, meta_names)

(
    X_train, Y1_train, Y2_train,
    X_val,   Y1_val,   Y2_val,
    X_test,  Y1_test,  Y2_test,
) = split_and_scale(X, Y1, Y2, seed=SEED)

train_loader = make_loader(X_train, Y1_train, Y2_train, BATCH_SIZE, shuffle=True)
val_loader   = make_loader(X_val,   Y1_val,   Y2_val,   128,        shuffle=False)
test_loader  = make_loader(X_test,  Y1_test,  Y2_test,  128,        shuffle=False)

print(f'Train : {len(X_train)} molecules')
print(f'Val   : {len(X_val)} molecules')
print(f'Test  : {len(X_test)} molecules')
print(f'Features        : {X_train.shape[1]}')
print(f'Fine labels     : {Y1.shape[1]}')
print(f'Meta labels     : {Y2.shape[1]}')
print(f'Hierarchy pairs : {len(violation_pairs)}')


## 7. Model and loss definitions

In [ ]:
class LocalBlock(nn.Module):
    def __init__(self, input_dim, global_dim, local_dim, n_labels, dropout):
        super().__init__()
        self.global_fc = nn.Sequential(
            nn.Linear(global_dim + input_dim, global_dim),
            nn.BatchNorm1d(global_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.transition = nn.Sequential(
            nn.Linear(global_dim, local_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.output = nn.Linear(local_dim, n_labels)

    def forward(self, x, A_G):
        A_G_next = self.global_fc(torch.cat([A_G, x], dim=1))
        A_L      = self.transition(A_G_next)
        P_L      = torch.sigmoid(self.output(A_L))
        return A_G_next, P_L


class HMCNF(nn.Module):
    def __init__(self, input_dim, n_fine, n_meta,
                 global_dim, local_dim, dropout, beta):
        super().__init__()
        self.beta    = beta
        self.n_total = n_fine + n_meta
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, global_dim),
            nn.BatchNorm1d(global_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.level1        = LocalBlock(input_dim, global_dim, local_dim, n_fine, dropout)
        self.level2        = LocalBlock(input_dim, global_dim, local_dim, n_meta, dropout)
        self.global_output = nn.Linear(global_dim, self.n_total)

    def forward(self, x):
        A_G       = self.input_proj(x)
        A_G, P_L1 = self.level1(x, A_G)
        A_G, P_L2 = self.level2(x, A_G)
        P_G       = torch.sigmoid(self.global_output(A_G))
        P_F       = self.beta * torch.cat([P_L1, P_L2], dim=1) + (1 - self.beta) * P_G
        return P_F, P_L1, P_L2, P_G


def binary_cross_entropy(P, Y, eps=1e-7):
    P = torch.clamp(P, eps, 1 - eps)
    return -torch.mean(Y * torch.log(P) + (1 - Y) * torch.log(1 - P))


def hierarchical_violation_penalty(P_L1, P_L2, pairs):
    total = torch.tensor(0.0, device=P_L1.device)
    for fi, mi in pairs:
        v = torch.clamp(P_L1[:, fi] - P_L2[:, mi], min=0.0)
        total = total + torch.mean(v ** 2)
    return total / max(len(pairs), 1)


def hmcn_loss(P_F, P_L1, P_L2, P_G, Y1, Y2, pairs, lambda_viol):
    Y_global       = torch.cat([Y1, Y2], dim=1)
    local_loss     = binary_cross_entropy(P_L1, Y1) + binary_cross_entropy(P_L2, Y2)
    global_loss    = binary_cross_entropy(P_G, Y_global)
    violation_loss = hierarchical_violation_penalty(P_L1, P_L2, pairs)
    return local_loss + global_loss + lambda_viol * violation_loss


## 8. Training helpers

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, pairs, lambda_viol, device):
    model.train()
    total_loss = 0.0
    for X_b, Y1_b, Y2_b in loader:
        X_b, Y1_b, Y2_b = X_b.to(device), Y1_b.to(device), Y2_b.to(device)
        optimizer.zero_grad()
        P_F, P_L1, P_L2, P_G = model(X_b)
        loss = hmcn_loss(P_F, P_L1, P_L2, P_G, Y1_b, Y2_b, pairs, lambda_viol)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def compute_val_loss(model, loader, pairs, lambda_viol, device):
    model.eval()
    total = 0.0
    for X_b, Y1_b, Y2_b in loader:
        X_b, Y1_b, Y2_b = X_b.to(device), Y1_b.to(device), Y2_b.to(device)
        P_F, P_L1, P_L2, P_G = model(X_b)
        total += hmcn_loss(P_F, P_L1, P_L2, P_G, Y1_b, Y2_b, pairs, lambda_viol).item()
    return total / len(loader)


@torch.no_grad()
def collect_predictions(model, loader, device):
    model.eval()
    fp, ft, mp, mt = [], [], [], []
    for X_b, Y1_b, Y2_b in loader:
        _, P_L1, P_L2, _ = model(X_b.to(device))
        fp.append(P_L1.cpu().numpy())
        ft.append(Y1_b.numpy())
        mp.append(P_L2.cpu().numpy())
        mt.append(Y2_b.numpy())
    return (np.vstack(fp), np.vstack(ft),
            np.vstack(mp), np.vstack(mt))


def compute_macro_roc_auc(probs, targets):
    return float(np.mean([
        roc_auc_score(targets[:, i], probs[:, i])
        for i in range(targets.shape[1])
        if targets[:, i].sum() > 0
    ]))


## 9. Sweep loop

Each config:
1. Builds and trains a fresh model
2. Loads best checkpoint
3. Calibrates thresholds on val set
4. Computes all metrics on test set via `hmcn_eval`
5. **Saves to CSV immediately** before moving to next config
6. Frees GPU memory

On restart, already-completed configs are skipped automatically.


In [ ]:
# Re-check completed configs in case CSV was updated since cell 4 ran
completed = get_completed_configs(CSV_PATH)
remaining  = [c for c in all_configs if config_key(c) not in completed]
n_total    = len(all_configs)

print(f'Starting sweep: {len(remaining)} remaining / {n_total} total')
print('─' * 70)

for run_idx, cfg in enumerate(remaining, start=1):

    lv = cfg['lambda_viol']
    lr = cfg['lr']
    wd = cfg['weight_decay']
    do = cfg['dropout']

    print(f'\n[{run_idx}/{len(remaining)}] '
          f'lambda={lv}  lr={lr:.0e}  wd={wd:.0e}  dropout={do}')

    # ── 1. Build model ────────────────────────────────────────────────────────
    model = HMCNF(
        input_dim  = X_train.shape[1],
        n_fine     = Y1_train.shape[1],
        n_meta     = Y2_train.shape[1],
        global_dim = GLOBAL_DIM,
        local_dim  = LOCAL_DIM,
        dropout    = do,
        beta       = BETA,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

    steps_per_epoch = len(train_loader)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0     = T_0_EPOCHS * steps_per_epoch,
        T_mult  = T_MULT,
        eta_min = ETA_MIN,
    )

    # ── 2. Train ──────────────────────────────────────────────────────────────
    best_val_auc      = 0.0
    best_model_state  = None
    train_loss_at_best = 0.0
    best_epoch        = 0
    patience_counter  = 0

    for epoch in range(1, EPOCHS + 1):
        train_loss  = train_one_epoch(
            model, train_loader, optimizer, scheduler,
            violation_pairs, lv, device
        )
        fp_v, ft_v, mp_v, mt_v = collect_predictions(model, val_loader, device)
        val_meta_auc = compute_macro_roc_auc(mp_v, mt_v)

        if val_meta_auc > best_val_auc:
            best_val_auc       = val_meta_auc
            best_model_state   = {k: v.cpu().clone()
                                  for k, v in model.state_dict().items()}
            train_loss_at_best = train_loss
            best_epoch         = epoch
            patience_counter   = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f'  Early stop @ epoch {epoch} | best epoch {best_epoch} | '
                      f'val AUC {best_val_auc:.4f}')
                break

    # ── 3. Load best checkpoint ───────────────────────────────────────────────
    model.load_state_dict(best_model_state)
    model.to(device)

    # ── 4. Calibrate thresholds on val ───────────────────────────────────────
    fp_v, ft_v, mp_v, mt_v = collect_predictions(model, val_loader, device)
    fine_thresholds = find_optimal_thresholds(fp_v, ft_v)
    meta_thresholds = find_optimal_thresholds(mp_v, mt_v)

    # ── 5. Evaluate on test set ───────────────────────────────────────────────
    fp_test, ft_test, mp_test, mt_test = collect_predictions(model, test_loader, device)

    test_loss_value = compute_test_loss(
        model, test_loader, violation_pairs, lv, device
    )

    metrics = compute_all_metrics(
        fine_probs      = fp_test,
        fine_true       = ft_test,
        meta_probs      = mp_test,
        meta_true       = mt_test,
        meta_thresholds = meta_thresholds,
        fine_thresholds = fine_thresholds,
        violation_pairs = violation_pairs,
        meta_names      = meta_names,
        fine_names      = fine_names,
        Y2_train        = Y2_train,
        test_loss       = test_loss_value,
    )

    print(f'  meta ROC AUC={metrics["roc_auc_12"]:.4f}  '
          f'PR AUC={metrics["pr_auc_12"]:.4f}  '
          f'F1={metrics["f1_macro_12"]:.4f}')

    # ── 6. Save to CSV immediately ────────────────────────────────────────────
    config_row = dict(
        experiment         = 'datasetA_sweep',
        param_name         = 'grid',
        param_value        = f'lv={lv}_lr={lr:.0e}_wd={wd:.0e}_do={do}',
        global_dim         = GLOBAL_DIM,
        local_dim          = LOCAL_DIM,
        dropout            = do,
        lr                 = lr,
        weight_decay       = wd,
        lambda_viol        = lv,
        beta               = BETA,
        batch_size         = BATCH_SIZE,
        seed               = SEED,
        best_epoch         = best_epoch,
        train_loss_at_best = train_loss_at_best,
        val_meta_roc_auc   = best_val_auc,
    )
    save_experiment(config_row, metrics, csv_path=CSV_PATH)

    # ── 7. Save model weights ─────────────────────────────────────────────────
    model_fname = (f'hmcn_sweep_lv{lv}_lr{lr:.0e}_wd{wd:.0e}_do{do}.pt')
    torch.save(best_model_state, model_fname)

    # ── 8. Free GPU memory ────────────────────────────────────────────────────
    del model, optimizer, scheduler, best_model_state
    gc.collect()
    torch.cuda.empty_cache()

print('\n' + '=' * 70)
print('SWEEP COMPLETE')
print(f'Results saved to: {CSV_PATH}')


## 10. Results summary

In [ ]:
results = pd.read_csv(CSV_PATH)
results = results[results['experiment'] == 'datasetA_sweep'].copy()

print(f'Completed runs: {len(results)}')
print()

# Sort by primary metric
results_sorted = results.sort_values('roc_auc_12', ascending=False)

print('Top 10 configs by Meta ROC AUC:')
cols = ['lambda_viol','lr','weight_decay','dropout',
        'best_epoch','val_meta_roc_auc',
        'roc_auc_12','pr_auc_12','f1_macro_12','balanced_accuracy_12']
print(results_sorted[cols].head(10).to_string(index=False))

print()
print('Best config overall:')
best = results_sorted.iloc[0]
print(f'  lambda_viol  : {best["lambda_viol"]}')
print(f'  lr           : {best["lr"]:.0e}')
print(f'  weight_decay : {best["weight_decay"]:.0e}')
print(f'  dropout      : {best["dropout"]}')
print(f'  → Meta ROC AUC : {best["roc_auc_12"]:.4f}')
print(f'  → Meta PR AUC  : {best["pr_auc_12"]:.4f}')
print(f'  → Meta F1      : {best["f1_macro_12"]:.4f}')
